<a href="https://colab.research.google.com/github/cyril2427/W26-AIGC5005--Midterm/blob/main/Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [65]:
import pandas as pd
import io
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from google.colab import files
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

In [87]:
uploaded = files.upload()

In [69]:
# 'risk-train.txt' is the uploaded filename
df = pd.read_csv(io.BytesIO(uploaded['risk-train (1).txt']), sep='\t', na_values='?')

# Check first few rows
df.head()

,ORDER_ID,CLASS,B_EMAIL,B_TELEFON,B_BIRTHDATE,FLAG_LRIDENTISCH,FLAG_NEWSLETTER,Z_METHODE,Z_CARD_ART,Z_CARD_VALID,...,FAIL_RPLZ,FAIL_RORT,FAIL_RPLZORTMATCH,SESSION_TIME,NEUKUNDE,AMOUNT_ORDER_PRE,VALUE_ORDER_PRE,DATE_LORDER,MAHN_AKT,MAHN_HOECHST
0,49917,no,yes,no,1/17/1973,yes,yes,check,NaN,5.2006,...,no,no,no,8,yes,0,0.00,NaN,NaN,NaN
1,49919,no,yes,yes,12/8/1970,no,no,credit_card,Visa,12.2007,...,yes,no,no,13,yes,0,0.00,NaN,NaN,NaN
2,49923,no,yes,no,4/3/1972,yes,no,check,NaN,12.2007,...,no,no,no,3,yes,0,0.00,NaN,NaN,NaN
3,49924,no,no,yes,8/1/1966,yes,no,check,NaN,1.2007,...,no,no,no,11,no,4,75.72,5/12/2002,0.0,0.0
4,49927,no,yes,yes,12/21/1969,yes,no,credit_card,Eurocard,12.2006,...,no,no,no,16,yes,0,0.00,NaN,NaN,NaN


In [70]:
df.shape

(30000, 44)

In [71]:
df.isnull().sum()

,0
ORDER_ID,0
CLASS,0
B_EMAIL,0
B_TELEFON,0
B_BIRTHDATE,2942
FLAG_LRIDENTISCH,0
FLAG_NEWSLETTER,0
Z_METHODE,0
Z_CARD_ART,18654
Z_CARD_VALID,0


In [72]:
df.describe()

,ORDER_ID,Z_CARD_VALID,VALUE_ORDER,AMOUNT_ORDER,ANUMMER_01,ANUMMER_02,ANUMMER_03,ANUMMER_04,ANUMMER_05,ANUMMER_06,ANUMMER_07,ANUMMER_08,ANUMMER_09,ANUMMER_10,SESSION_TIME,AMOUNT_ORDER_PRE,VALUE_ORDER_PRE,MAHN_AKT,MAHN_HOECHST
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,7853.000000,3198.000000,1332.000000,541.000000,206.000000,95.000000,34.000000,7.000000,0.0,30000.000000,30000.000000,30000.000000,14144.000000,14144.000000
mean,25032.612933,6.680567,43.968058,1.442200,331860.254033,333235.324462,330570.160725,332915.267267,330867.528651,328556.514563,320592.989474,334612.382353,206067.285714,NaN,8.577900,0.606700,29.875652,0.129596,0.425269
std,14422.306098,3.468332,35.709431,0.921496,147310.846584,147581.876885,146720.473881,148010.492904,146836.981054,146807.315275,158345.184258,158550.890657,140780.337373,NaN,3.863448,0.765616,57.382732,0.432493,0.771451
min,1.000000,1.200500,5.200000,1.000000,100061.000000,100061.000000,100061.000000,100061.000000,100124.000000,100767.000000,101371.000000,104921.000000,102681.000000,NaN,1.000000,0.000000,0.000000,0.000000,0.000000
25%,12518.750000,3.200700,17.990000,1.000000,204159.000000,204258.000000,204258.000000,204242.000000,204935.000000,204386.250000,200490.500000,202266.500000,106271.500000,NaN,6.000000,0.000000,0.000000,0.000000,0.000000
50%,25095.500000,6.200700,34.500000,1.000000,401241.000000,401241.000000,401219.500000,401402.000000,401564.000000,400526.000000,400767.000000,402111.000000,109102.000000,NaN,9.000000,0.000000,0.000000,0.000000,0.000000
75%,37520.250000,10.200500,57.800000,2.000000,407703.000000,407699.000000,407703.000000,407741.000000,407491.000000,408333.500000,408899.000000,409531.000000,306711.000000,NaN,11.000000,1.000000,38.470000,0.000000,1.000000
max,50000.000000,12.200700,361.200000,9.000000,609725.000000,609725.000000,609725.000000,609725.000000,609725.000000,609500.000000,608997.000000,609500.000000,404723.000000,NaN,24.000000,6.000000,1047.800000,3.000000,3.000000


In [73]:
# Dropping the following, dataset has ~30,000 rows, these columns are ~50% to ~100% missing

df.drop(columns=['ANUMMER_02','ANUMMER_03','ANUMMER_04','ANUMMER_05','ANUMMER_06','ANUMMER_07',
                 'ANUMMER_08','ANUMMER_09','ANUMMER_10','Z_CARD_ART','DATE_LORDER','MAHN_AKT','MAHN_HOECHST'], inplace=True)

In [74]:
# Droppig Identifiers (High cardinality, no predictive power)
df.drop(columns=['ORDER_ID','B_EMAIL','B_TELEFON'], inplace=True)

In [75]:
# Dropping Personal / sensitive info
df.drop(columns=['Z_LAST_NAME'], inplace=True)

In [76]:
# Handling B_BIRTHDATE (2942 missing)

df['B_BIRTHDATE'] = pd.to_datetime(df['B_BIRTHDATE'], errors='coerce')
df['AGE'] = (pd.to_datetime('today') - df['B_BIRTHDATE']).dt.days // 365

df['AGE'].fillna(df['AGE'].median(), inplace=True)

df.drop(columns=['B_BIRTHDATE'], inplace=True)

/tmp/ipykernel_2461/1570202984.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['AGE'].fillna(df['AGE'].median(), inplace=True)


In [77]:
# Handling TIME_ORDER

# Convert TIME_ORDER to datetime
df['TIME_ORDER'] = pd.to_datetime(df['TIME_ORDER'], format='%H:%M', errors='coerce')

# Convert to minutes since midnight
df['TIME_ORDER_MIN'] = df['TIME_ORDER'].dt.hour * 60 + df['TIME_ORDER'].dt.minute

# Fill missing with median
df['TIME_ORDER_MIN'].fillna(df['TIME_ORDER_MIN'].median(), inplace=True)
df.drop(columns=['TIME_ORDER'], inplace=True)

/tmp/ipykernel_2461/992641346.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TIME_ORDER_MIN'].fillna(df['TIME_ORDER_MIN'].median(), inplace=True)


In [78]:
df['TIME_ORDER_MIN'].isnull().sum()

np.int64(0)

### Encode Target Variable

In [79]:
df['CLASS'] = df['CLASS'].map({'no': 0, 'yes': 1})

### Encode categorical features

In [80]:
df.select_dtypes(include='object').columns

Index(['FLAG_LRIDENTISCH', 'FLAG_NEWSLETTER', 'Z_METHODE', 'WEEKDAY_ORDER',
       'CHK_LADR', 'CHK_RADR', 'CHK_KTO', 'CHK_CARD', 'CHK_COOKIE', 'CHK_IP',
       'FAIL_LPLZ', 'FAIL_LORT', 'FAIL_LPLZORTMATCH', 'FAIL_RPLZ', 'FAIL_RORT',
       'FAIL_RPLZORTMATCH', 'NEUKUNDE'],
      dtype='object')

In [81]:
binary_cols = ['FLAG_LRIDENTISCH','FLAG_NEWSLETTER','CHK_LADR','CHK_RADR',
               'CHK_KTO','CHK_CARD','CHK_COOKIE','CHK_IP',
               'FAIL_LPLZ','FAIL_LORT','FAIL_LPLZORTMATCH','FAIL_RPLZ',
               'FAIL_RORT','FAIL_RPLZORTMATCH','NEUKUNDE']

df[binary_cols] = df[binary_cols].apply(lambda x: x.map({'no':0,'yes':1}))

In [82]:
df = pd.get_dummies(df, columns=['Z_METHODE','WEEKDAY_ORDER'], drop_first=True)

In [83]:
df.dtypes.value_counts()

,count
int64,20
bool,9
float64,5


### Train-test split

In [84]:
X = df.drop(columns=['CLASS'])
y = df['CLASS']



X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [85]:


sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)


In [86]:


# Train model
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced_subsample',
    random_state=42
)
rf.fit(X_train, y_train)

# Predict with adjusted threshold
y_prob = rf.predict_proba(X_test)[:,1]
y_pred = (y_prob >= 0.2).astype(int)

# Evaluate
print(classification_report(y_test, y_pred))
print("F1-score for 'yes' class:", f1_score(y_test, y_pred, pos_label=1))

              precision    recall  f1-score   support

           0       0.95      0.98      0.96      5651
           1       0.21      0.10      0.14       349

    accuracy                           0.93      6000
   macro avg       0.58      0.54      0.55      6000
weighted avg       0.90      0.93      0.91      6000

F1-score for 'yes' class: 0.13846153846153847
